# Module 14 — Agent Skills

> **SDKs:** `pydantic`, `dataclasses`, `re`

| Part | Topic |
|------|-------|
| **1** | Anatomy of a Skill — inputs, outputs, instructions |
| **2** | Skills vs MCP — when to use each |
| **3** | Skill Libraries & Routing — dynamic skill activation |


---
## Part 1 — Anatomy of a Skill

A Skill is a composable capability that an agent can activate. Unlike MCP Tools (which are API calls), Skills can contain complex multi-step reasoning instructions.

In [ ]:
from dataclasses import dataclass, field
from typing import Literal, Optional
from pydantic import BaseModel

@dataclass
class SkillInput:
    """Strictly typed input schema for a skill."""
    service: str
    time_window_minutes: int = 30

class SkillOutput(BaseModel):
    """Strictly typed output schema — the agent must produce this shape."""
    hypothesis: str
    confidence: Literal["LOW", "MEDIUM", "HIGH"]
    evidence_ids: list[str]
    recommended_action: str

@dataclass
class Skill:
    """
    A composable agent capability.
    Contains: typed I/O, system prompt fragment, tool scope, and success criteria.
    """
    name: str
    description: str
    input_schema: type
    output_schema: type
    system_prompt_fragment: str
    allowed_tools: list[str]
    max_steps: int = 10

    def activate(self, inputs: dict) -> dict:
        """Simulates skill execution: reads inputs, follows instructions, returns typed output."""
        print(f"  [Skill: {self.name}] Activating with inputs: {inputs}")
        print(f"  [Skill] Allowed tools: {self.allowed_tools}")
        print(f"  [Skill] Max steps: {self.max_steps}")
        # Simulate execution
        return SkillOutput(
            hypothesis=f"3DS redirect broken in {inputs.get('service','unknown')} after recent deployment",
            confidence="HIGH",
            evidence_ids=["EV-001", "EV-003"],
            recommended_action="Propose revert via feature-flag — requires HITL approval",
        ).model_dump()

# ─── Define a skill ───────────────────────────────────────────────────────────
diagnose_skill = Skill(
    name="diagnose_service_incident",
    description="Investigates a service incident using telemetry and deployment data",
    input_schema=SkillInput,
    output_schema=SkillOutput,
    system_prompt_fragment=(
        "You are an incident investigator. Use read-only tools to collect evidence. "
        "Never propose mutations. Always cite evidence IDs in your output."
    ),
    allowed_tools=["query_metrics", "search_logs", "get_deployment", "read_runbook"],
    max_steps=8,
)

print("🧩  Skill Anatomy Demo")
print("=" * 60)
print(f"  Skill: {diagnose_skill.name}")
print(f"  Description: {diagnose_skill.description}")
print(f"  Allowed tools: {diagnose_skill.allowed_tools}")
print(f"  Max steps: {diagnose_skill.max_steps}")
print(f"  Prompt fragment: '{diagnose_skill.system_prompt_fragment[:60]}...'")
print()

result = diagnose_skill.activate({"service": "checkout-ui", "time_window_minutes": 30})
print(f"\n  Skill output (typed):")
for k, v in result.items():
    print(f"    {k:<22} : {v}")


🧩  Skill Anatomy Demo
  Skill: diagnose_service_incident
  Description: Investigates a service incident using telemetry and deployment data
  Allowed tools: ['query_metrics', 'search_logs', 'get_deployment', 'read_runbook']
  Max steps: 8
  Prompt fragment: 'You are an incident investigator. Use read-only tools to c...'

  [Skill: diagnose_service_incident] Activating with inputs: {'service': 'checkout-ui', 'time_window_minutes': 30}
  [Skill] Allowed tools: ['query_metrics', 'search_logs', 'get_deployment', 'read_runbook']
  [Skill] Max steps: 8

  Skill output (typed):
    hypothesis             : 3DS redirect broken in checkout-ui after recent deployment
    confidence             : HIGH
    evidence_ids           : ['EV-001', 'EV-003']
    recommended_action     : Propose revert via feature-flag — requires HITL approval


---
## Part 2 — Skills vs MCP

MCP Tools are point-in-time API calls (read data, write data). Skills are stateful, multi-step reasoning procedures. The two are complementary — Skills *use* MCP Tools internally.

In [ ]:
from dataclasses import dataclass

@dataclass
class ComparisonRow:
    dimension: str
    mcp_tool: str
    skill: str

comparison = [
    ComparisonRow("Definition",       "Single function with JSON schema",         "Multi-step reasoning procedure"),
    ComparisonRow("State",            "Stateless — input → output",                "Stateful — tracks evidence across steps"),
    ComparisonRow("Instructions",     "None (just schema)",                        "System prompt fragment"),
    ComparisonRow("Tools",            "IS the tool",                               "USES MCP tools internally"),
    ComparisonRow("Reuse",            "By name in tool_choice",                    "Activated by router based on intent"),
    ComparisonRow("Best for",         "Atomic data retrieval/mutation",            "Complex investigation or generation"),
    ComparisonRow("Example",          "query_metrics(service, window)",            "diagnose_incident(service, window)"),
]

print("⚖️  Skills vs MCP: Comparison")
print("=" * 75)
print(f"  {'Dimension':<18} {'MCP Tool':<33} {'Skill'}")
print(f"  {'─'*18} {'─'*33} {'─'*30}")
for r in comparison:
    print(f"  {r.dimension:<18} {r.mcp_tool:<33} {r.skill}")


⚖️  Skills vs MCP: Comparison
  Dimension          MCP Tool                           Skill
  ────────────────── ───────────────────────────────── ──────────────────────────────
  Definition         Single function with JSON schema   Multi-step reasoning procedure
  State              Stateless — input → output         Stateful — tracks evidence across steps
  Instructions       None (just schema)                 System prompt fragment
  Tools              IS the tool                        USES MCP tools internally
  Reuse              By name in tool_choice             Activated by router based on intent
  Best for           Atomic data retrieval/mutation     Complex investigation or generation
  Example            query_metrics(service, window)     diagnose_incident(service, window)


---
## Part 3 — Skill Libraries & Dynamic Routing

A Skill Library is a registry of available skills. The router maps user intent to the correct skill using lightweight keyword matching (not an LLM).

In [ ]:
import re
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class SkillRegistration:
    skill: Skill
    trigger_patterns: list[str]  # regex patterns that activate this skill
    priority: int = 1

class SkillLibrary:
    def __init__(self):
        self._skills: list[SkillRegistration] = []

    def register(self, reg: SkillRegistration):
        self._skills.append(reg)
        self._skills.sort(key=lambda r: -r.priority)

    def route(self, user_intent: str) -> Optional[Skill]:
        """Deterministic pattern matching — no LLM needed."""
        lower = user_intent.lower()
        for reg in self._skills:
            for pattern in reg.trigger_patterns:
                if re.search(pattern, lower):
                    print(f"  [Library] Pattern '{pattern}' matched → activating '{reg.skill.name}'")
                    return reg.skill
        print(f"  [Library] No skill matched intent: '{user_intent[:50]}'")
        return None

# ─── Build a library ─────────────────────────────────────────────────────────
skill_library = SkillLibrary()

skill_library.register(SkillRegistration(diagnose_skill, 
    ["incident", "outage", "conversion.*down", "error.*rate", "failing"], priority=2))

refund_skill = Skill("process_refund", "Process customer refund via Stripe",
    SkillInput, SkillOutput, "Process refund. Validate idempotency.", ["stripe_api", "billing_db"], max_steps=4)
skill_library.register(SkillRegistration(refund_skill, 
    ["refund", "charged.*twice", "billing.*error"], priority=1))

faq_skill = Skill("answer_faq", "Answer general FAQ from knowledge base",
    SkillInput, SkillOutput, "Answer from KB only. No tools.", ["search_knowledge_base"], max_steps=2)
skill_library.register(SkillRegistration(faq_skill,
    ["how.*do", "what.*is", "how.*reset"], priority=0))

# ─── Route test queries ────────────────────────────────────────────────────────
queries = [
    "Checkout conversion is down 38% — what's happening?",
    "I was charged twice on my invoice",
    "How do I reset my API key?",
    "Tell me something interesting",
]

print("📚  Skill Library Routing Demo")
print("=" * 60)
for q in queries:
    print(f"\n  Query: '{q[:55]}'")
    skill = skill_library.route(q)
    if skill:
        print(f"  → Skill: {skill.name}")
    else:
        print(f"  → No match — escalate to human support")


📚  Skill Library Routing Demo

  Query: 'Checkout conversion is down 38% — what's happening?'
  [Library] Pattern 'conversion.*down' matched → activating 'diagnose_service_incident'
  → Skill: diagnose_service_incident

  Query: 'I was charged twice on my invoice'
  [Library] Pattern 'charged.*twice' matched → activating 'process_refund'
  → Skill: process_refund

  Query: 'How do I reset my API key?'
  [Library] Pattern 'how.*reset' matched → activating 'answer_faq'
  → Skill: answer_faq

  Query: 'Tell me something interesting'
  [Library] No skill matched intent: 'Tell me something interesting'
  → No match — escalate to human support
